## Test loss behavior of BabyGPT

In [8]:
import os
import math
from tqdm import tqdm
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM,DataCollatorForLanguageModeling
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import PeftModel, LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import numpy as np
from torch.nn import functional as F
from torch.utils.data import DataLoader

In [ ]:
device = "cuda" if torch.cuda.is_available() else 'cpu'
device

NameError: name 'torch' is not defined

## Connect google drive, hugging face, load model

In [5]:
from google.colab import drive
import os
drive.mount('/content/drive')

NOTEBOOK_DIR = "/content/drive/MyDrive/Colab_Notebooks"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from huggingface_hub import login
hf_token = "" # removed in public repo
login(hf_token, add_to_git_credential=True)

In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    "littleBallOfFur/baby-gpt-base",
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "littleBallOfFur/baby-gpt-base",
    trust_remote_code=True,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/584 [00:00<?, ?B/s]

babygpt_model.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/littleBallOfFur/baby-gpt-base:
- babygpt_model.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/652M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [7]:
tokenizer.pad_token = tokenizer.eos_token
model.train()

BabyGPTForCausalLM(
  (transformer): ModuleDict(
    (wte): Embedding(50304, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50304, bias=False)
)

### dataset

In [14]:
DATASET_NAME = "karpathy/tinystories-gpt4-clean"
raw_dataset = load_dataset(DATASET_NAME)

split_1 = raw_dataset['train'].select(range(2000,3000)).train_test_split(test_size=0.2, seed=42)
train_dataset = split_1["train"]
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)
test_dataset = split_2["test"]
val_dataset = split_2["train"]
len(train_dataset), len(val_dataset),len(test_dataset)

(800, 100, 100)

In [16]:
MAX_TOKENS = 256
def keep_short(batch):
    texts = batch["text"]
    lengths = [
        len(ids)
        for ids in tokenizer(texts, add_special_tokens=True)["input_ids"]
    ]
    return [n <= MAX_TOKENS for n in lengths]

train_dataset = train_dataset.filter(keep_short, batched=True)
small_dataset = val_dataset.filter(keep_short, batched = True)
test_dataset = test_dataset.filter(keep_short, batched = True)

len(train_dataset), len(val_dataset),len(test_dataset)


Filter:   0%|          | 0/700 [00:00<?, ? examples/s]

Filter:   0%|          | 0/91 [00:00<?, ? examples/s]

(700, 100, 91)

In [18]:
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_TOKENS,
        padding=False,
    )

tokenized_train = train_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_valid = val_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=val_dataset.column_names,
)

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [20]:
BATCH_SIZE = 4
collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

train_loader = DataLoader(
    tokenized_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
)

valid_loader = DataLoader(
    tokenized_valid,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
)

In [ ]:
temp = next(iter(train_loader))
# temp.attention_mask, temp.input_ids, temp.labels

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

###

### loop

In [36]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    betas=(0.9, 0.95),
    weight_decay=0.01,
)

max_steps = 20
log_every = 1
eval_every = 3

step = 0
running_loss = 0.0

model.train()

for batch in tqdm(train_loader,desc="training"):
    batch = {k: v.to(device) for k, v in batch.items()}

    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"],
    )

    loss = outputs.loss

    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()

    running_loss += loss.item()
    step += 1

    if step % log_every == 0:
        # avg_loss = running_loss / log_every
        print(f"step {step:04d} | train loss {loss.item():.4f}")
        running_loss = 0.0
    
    if step % eval_every ==0:
        model.eval()
        with torch.no_grad():
            test_batch = next(iter(valid_loader))
            loss = model(**test_batch).loss
            print(f"step {step:04d} | validation loss {loss.item():.4f}")

    if step >= max_steps:
        break

training:   1%|          | 1/175 [00:15<44:29, 15.34s/it]

step 0001 | train loss 2.2166


training:   1%|          | 2/175 [00:26<36:41, 12.72s/it]

step 0002 | train loss 2.1104
step 0003 | train loss 2.0483


training:   2%|▏         | 3/175 [00:45<45:21, 15.82s/it]

...

step 0018 | validation loss 1.9995


training:  11%|█         | 19/175 [04:33<35:03, 13.48s/it]

step 0019 | train loss 1.9478


training:  11%|█         | 19/175 [04:48<39:25, 15.16s/it]

step 0020 | train loss 1.8559


In [35]:
def eval_loss(model, dataloader,max_batches = 4):
    model.eval()

    total_loss = 0.0
    total_batches = 0

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )

        total_loss += outputs.loss.item()
        total_batches += 1
        if total_batches>= max_batches:
            break

    model.train()
    return total_loss / total_batches


val_loss = eval_loss(model, valid_loader,max_batches = 4)
print("valid loss:", val_loss)
print("valid perplexity:", torch.exp(torch.tensor(val_loss)).item())

valid loss: 2.063559740781784
valid perplexity: 7.8739495277404785


In [48]:
# torch.manual_seed(1337)
model.eval()
prompt = "Once upon a time there was a squirrel "
inputs = tokenizer(prompt, return_tensors='pt').to(device)
output = model.generate(**inputs, max_new_tokens=32,do_sample=True,top_k=50)
print(tokenizer.decode(output[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time there was a squirrel iced up inside a big garden. It liked to feel lonely and lost. One day he went for a walk and saw a little dog. The dog was sad


In [39]:
base_model = AutoModelForCausalLM.from_pretrained(
    "littleBallOfFur/baby-gpt-base",
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

In [44]:
torch.manual_seed(1337)
base_model.eval()
prompt = "Once upon a time there was a squirrel "
inputs = tokenizer(prompt, return_tensors='pt').to(device)
output = base_model.generate(**inputs, max_new_tokens=32,do_sample=True,top_k=50)
print(tokenizer.decode(output[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time there was a squirrel _________ for himself.
- What is the relationship among squirrel and man?
- Why is the man a _________?
- What makes an


In [41]:
gpt2_model = AutoModelForCausalLM.from_pretrained(
    "gpt2"
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [46]:
torch.manual_seed(1337)
gpt2_model.eval()
prompt = "Once upon a time there was a squirrel "
inputs = tokenizer(prompt, return_tensors='pt').to(device)
output = gpt2_model.generate(**inputs, max_new_tokens=32,do_sample=True,top_k=50)
print(tokenizer.decode(output[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time there was a squirrel  having a breakfast with  The Old Ones  (and  The King ) at my  home or around another  place. This was around 2000 years ago
